# Demo: Resolved Market Metadata -> 5-Minute Probability Series -> SQLite

This notebook demonstrates a compact end-to-end workflow for a **resolved** Polymarket market:

1. Download broad market metadata.
2. Filter to closed binary markets with a final resolved outcome.
3. Pick a liquid candidate whose trade history is still practical to download.
4. Convert full trade history into a `5m` panel of `Yes` probabilities.
5. Save the selected market metadata, raw trades, and `5m` probability series into SQLite.

The goal is not to build the final dataset here. The goal is to show a reproducible path from raw Polymarket metadata to a database-backed `market_id x timestamp` panel.


## Step 0: Environment Setup

This cell makes the notebook work from either the repository root or the `examples/` directory.


In [2]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "clients").exists() else cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/sneddy/research/polymarket_research


In [3]:
from __future__ import annotations

import ast
import math
import sqlite3
from typing import Any

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

from clients.gamma_client import GammaClient
from collectors.markets_collector import MarketsCollector
from collectors.trades_collector import TradesCollector


## Step 1: Configure The Demo

A few knobs keep the demo practical:

- download a broad recent market universe first,
- keep only binary markets with a resolved `0/1`-like final state and meaningful historical volume,
- enrich the strongest candidates with official market tags, because recent bulk metadata often omits top-level `category`,
- keep only domains that match the research focus: politics, geopolitics, finance/economy, technology, and crypto,
- prefer a candidate whose trade history is large enough to be interesting but not so large that the notebook becomes painful to run.


In [ ]:
MIN_CREATED_AT = "2025-01-01T00:00:00Z"
MIN_RESOLVED_VOLUME = 100_000.0
MAX_METADATA_PAGES = 10
TAG_ENRICH_SCAN_LIMIT = 120
CANDIDATE_SCAN_LIMIT = 40
MAX_TRADE_COUNT = 20_000
TARGET_TAG_MAP = {
    "politics": {"Politics", "US Politics", "US-current-affairs", "Elections"},
    "geopolitics": {"Global Politics", "Geopolitics", "Middle East", "Russia-Ukraine"},
    "finance_economy": {"Business", "Finance", "Economics", "Fed", "Inflation", "Recession"},
    "technology": {"Tech", "AI"},
    "crypto": {"Crypto", "Bitcoin", "Ethereum", "Solana", "NFTs"},
}
ALLOWED_TAGS = {tag for values in TARGET_TAG_MAP.values() for tag in values}
DB_PATH = REPO_ROOT / "db" / "demo_resolved_probability_series.sqlite"

print(f"SQLite output: {DB_PATH}")
print("Target research domains:", ", ".join(TARGET_TAG_MAP.keys()))
print("Allowed market tags:", sorted(ALLOWED_TAGS))


## Step 2: Helper Functions

These helpers do three things:

- normalize Gamma metadata into a clean candidate table,
- select one resolved/liquid market with accessible trade history,
- transform raw trade history into a `5m` `Yes`-probability panel.


In [ ]:
def _to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def _parse_list(value: Any) -> list[Any] | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return ast.literal_eval(text)
        except Exception:
            return None
    return None


def _parse_binary_prices(value: Any) -> list[float] | None:
    parsed = _parse_list(value)
    if not isinstance(parsed, list) or len(parsed) != 2:
        return None
    try:
        out = [float(x) for x in parsed]
    except Exception:
        return None
    if not all(math.isfinite(x) for x in out):
        return None
    return out


def _parse_binary_outcomes(value: Any) -> list[str] | None:
    parsed = _parse_list(value)
    if not isinstance(parsed, list) or len(parsed) != 2:
        return None
    out = [str(x).strip() for x in parsed]
    return out if {x.lower() for x in out} == {"yes", "no"} else None


def _resolved_outcome_from_prices(prices: list[float], outcomes: list[str]) -> tuple[str, float] | tuple[None, None]:
    if len(prices) != 2 or len(outcomes) != 2:
        return None, None
    if abs(sum(prices) - 1.0) > 1e-3:
        return None, None
    winner_idx = int(np.argmax(prices))
    winner_prob = float(prices[winner_idx])
    if winner_prob < 0.99:
        return None, None
    return outcomes[winner_idx], winner_prob


def summarize_bulk_categories(markets_df: pd.DataFrame) -> pd.DataFrame:
    if "category" not in markets_df.columns:
        return pd.DataFrame(
            [
                {
                    "category": "<missing in bulk metadata>",
                    "market_count": int(len(markets_df)),
                    "selected_for_demo": False,
                }
            ]
        )

    category_series = markets_df["category"].astype("string").str.strip()
    summary = (
        category_series.fillna("<NULL>")
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="market_count")
        .sort_values(["market_count", "category"], ascending=[False, True])
        .reset_index(drop=True)
    )
    return summary.assign(selected_for_demo=False)


def _normalize_tag_labels(tag_payload: Any) -> list[str]:
    if not isinstance(tag_payload, list):
        return []
    labels: list[str] = []
    for item in tag_payload:
        if not isinstance(item, dict):
            continue
        label = item.get("label")
        if label is None:
            continue
        clean = str(label).strip()
        if clean:
            labels.append(clean)
    return labels


def build_candidate_pool(markets_df: pd.DataFrame) -> pd.DataFrame:
    work = markets_df.copy()
    work["closed_bool"] = _to_bool(work["closed"]) if "closed" in work.columns else False
    work["active_bool"] = _to_bool(work["active"]) if "active" in work.columns else False
    work["volume_num"] = pd.to_numeric(work.get("volume"), errors="coerce").fillna(0.0)
    work["created_at"] = pd.to_datetime(work.get("created_at"), utc=True, errors="coerce")
    work["end_date"] = pd.to_datetime(work.get("end_date"), utc=True, errors="coerce")
    work["parsed_prices"] = work.get("outcome_prices").map(_parse_binary_prices)
    work["parsed_outcomes"] = work.get("outcomes").map(_parse_binary_outcomes)

    final_outcomes: list[str | None] = []
    final_yes_probabilities: list[float | None] = []
    resolved_like: list[bool] = []
    for prices, outcomes in zip(work["parsed_prices"], work["parsed_outcomes"], strict=False):
        if prices is None or outcomes is None:
            final_outcomes.append(None)
            final_yes_probabilities.append(None)
            resolved_like.append(False)
            continue
        winner, _ = _resolved_outcome_from_prices(prices, outcomes)
        final_outcomes.append(winner)
        final_yes_probabilities.append(float(prices[outcomes.index("Yes")]) if winner is not None else None)
        resolved_like.append(winner is not None)

    work["final_outcome"] = final_outcomes
    work["final_yes_probability"] = final_yes_probabilities
    work["resolved_like"] = resolved_like
    work["has_clob_token_ids"] = work.get("clob_token_ids").notna() if "clob_token_ids" in work.columns else False

    mask = (
        work["closed_bool"]
        & work["resolved_like"]
        & work["has_clob_token_ids"]
        & (work["volume_num"] >= float(MIN_RESOLVED_VOLUME))
    )

    cols = [
        "id",
        "condition_id",
        "slug",
        "question",
        "created_at",
        "end_date",
        "volume_num",
        "final_outcome",
        "final_yes_probability",
        "outcomes",
        "outcome_prices",
    ]
    available_cols = [c for c in cols if c in work.columns]
    return work.loc[mask, available_cols].sort_values(["volume_num", "created_at"], ascending=[False, False]).reset_index(drop=True)


def enrich_candidate_tags(candidate_df: pd.DataFrame, gamma_client: GammaClient) -> pd.DataFrame:
    enriched_rows: list[dict[str, Any]] = []
    for _, row in candidate_df.head(TAG_ENRICH_SCAN_LIMIT).iterrows():
        payload = gamma_client.get_market_tags(row["id"])
        tag_labels = _normalize_tag_labels(payload)
        matched_tags = sorted(set(tag_labels).intersection(ALLOWED_TAGS))
        matched_domains = sorted(
            domain for domain, tags in TARGET_TAG_MAP.items() if set(tag_labels).intersection(tags)
        )
        enriched = row.to_dict()
        enriched["tag_labels"] = tag_labels
        enriched["matched_tags"] = matched_tags
        enriched["matched_domains"] = matched_domains
        enriched["selected_for_demo"] = bool(matched_domains)
        enriched_rows.append(enriched)

    if not enriched_rows:
        raise RuntimeError("No candidate markets were available for tag enrichment.")

    return pd.DataFrame(enriched_rows)


def pick_demo_market(candidate_df: pd.DataFrame, trades: TradesCollector) -> pd.Series:
    attempts: list[dict[str, Any]] = []
    for _, row in candidate_df.head(CANDIDATE_SCAN_LIMIT).iterrows():
        estimate = None
        error = None
        try:
            estimate = trades.estimate_trade_count(str(row["condition_id"]))
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"

        attempts.append(
            {
                "slug": row["slug"],
                "matched_domains": row.get("matched_domains"),
                "matched_tags": row.get("matched_tags"),
                "volume_num": float(row["volume_num"]),
                "trade_count_estimate": estimate,
                "selection_error": error,
            }
        )

        if estimate is not None and 1 <= int(estimate) <= int(MAX_TRADE_COUNT):
            selected = row.copy()
            selected["trade_count_estimate"] = int(estimate)
            return selected

    scanned = pd.DataFrame(attempts)
    display(scanned.head(20))
    raise RuntimeError(
        "Could not find a resolved candidate with accessible trade history under the configured trade-count cap. "
        "Raise MAX_TRADE_COUNT or widen the metadata scan."
    )


def build_yes_probability_series_5m(trades_df: pd.DataFrame, market_row: pd.Series) -> pd.DataFrame:
    work = trades_df.copy()
    work["timestamp_utc"] = pd.to_datetime(work["timestamp_utc"], utc=True, errors="coerce")
    work["price"] = pd.to_numeric(work["price"], errors="coerce")
    work["size"] = pd.to_numeric(work["size"], errors="coerce")
    work["outcome"] = work["outcome"].astype("string")
    outcome_norm = work["outcome"].str.strip().str.lower()
    work["yes_probability"] = np.where(
        outcome_norm.eq("yes"),
        work["price"],
        np.where(outcome_norm.eq("no"), 1.0 - work["price"], np.nan),
    )
    work = work.dropna(subset=["timestamp_utc", "yes_probability"]).sort_values("timestamp_utc").reset_index(drop=True)
    if work.empty:
        raise RuntimeError("No usable Yes/No trade history remained after normalization.")

    agg = (
        work.set_index("timestamp_utc")
        .groupby(pd.Grouper(freq="5min"))
        .agg(
            last_yes_probability=("yes_probability", "last"),
            trade_count=("transaction_hash", "size"),
            total_size=("size", "sum"),
            last_trade_price=("price", "last"),
        )
    )

    grid = pd.date_range(
        start=work["timestamp_utc"].min().floor("5min"),
        end=work["timestamp_utc"].max().ceil("5min"),
        freq="5min",
        tz="UTC",
    )

    panel = pd.DataFrame(index=grid).join(agg, how="left")
    panel.index.name = "timestamp_utc"
    panel["yes_probability"] = panel["last_yes_probability"].ffill()
    panel["trade_count"] = panel["trade_count"].fillna(0).astype(int)
    panel["total_size"] = panel["total_size"].fillna(0.0)
    panel["observed_trade"] = panel["last_yes_probability"].notna()
    panel["market_id"] = market_row.get("id")
    panel["condition_id"] = market_row["condition_id"]
    panel["market_slug"] = market_row["slug"]
    panel["question"] = market_row["question"]
    panel["tag_labels"] = ", ".join(market_row.get("tag_labels", [])) if isinstance(market_row.get("tag_labels"), list) else market_row.get("tag_labels")
    panel["matched_domains"] = ", ".join(market_row.get("matched_domains", [])) if isinstance(market_row.get("matched_domains"), list) else market_row.get("matched_domains")
    panel["final_outcome"] = market_row["final_outcome"]
    panel["final_yes_probability"] = market_row["final_yes_probability"]
    return panel.reset_index()


def _sqlite_safe_value(value: Any) -> Any:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, pd.Timestamp):
        ts = value.tz_convert("UTC") if value.tzinfo is not None else value.tz_localize("UTC")
        return ts.strftime("%Y-%m-%dT%H:%M:%SZ")
    if isinstance(value, list):
        return ", ".join(str(v) for v in value)
    return value


def write_demo_sqlite(
    db_path: Path,
    market_row: pd.Series,
    trades_df: pd.DataFrame,
    probability_df: pd.DataFrame,
) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)

    metadata_df = market_row.to_frame().T.copy()
    metadata_df = metadata_df.apply(lambda col: col.map(_sqlite_safe_value))

    trades_out = trades_df.copy()
    probability_out = probability_df.copy()
    for df in (trades_out, probability_out):
        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = pd.to_datetime(df[col], utc=True, errors="coerce").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            elif df[col].map(lambda value: isinstance(value, list)).any():
                df[col] = df[col].map(_sqlite_safe_value)

    with sqlite3.connect(db_path) as conn:
        metadata_df.to_sql("selected_market_metadata", conn, if_exists="replace", index=False)
        trades_out.to_sql("selected_market_trades", conn, if_exists="replace", index=False)
        probability_out.to_sql("selected_market_probability_5m", conn, if_exists="replace", index=False)



## Step 3: Download Metadata And Build A Resolved Candidate Pool

The metadata download is intentionally broader than the final selection. We want a realistic shortlist of closed markets and then let the notebook choose one candidate automatically.


In [ ]:
gamma = GammaClient()
markets_collector = MarketsCollector(gamma)
trades_collector = TradesCollector(gamma)

report = markets_collector.download_market_meta(
    include_active=True,
    include_inactive=True,
    limit=200,
    max_pages=MAX_METADATA_PAGES,
    min_created_at=MIN_CREATED_AT,
    show_progress=True,
    estimate_total=False,
    frame_type="pandas",
)

markets_df = report["markets"]
bulk_category_summary_df = summarize_bulk_categories(markets_df)
raw_candidate_df = build_candidate_pool(markets_df)
tag_enriched_candidate_df = enrich_candidate_tags(raw_candidate_df, gamma)
candidate_df = tag_enriched_candidate_df.loc[tag_enriched_candidate_df["selected_for_demo"]].reset_index(drop=True)

print(f"Downloaded markets: {len(markets_df):,}")
print("Bulk metadata category availability:")
display(bulk_category_summary_df)
print(f"Resolved binary candidates before tag filtering: {len(raw_candidate_df):,}")
print(f"Candidates enriched with official tags: {len(tag_enriched_candidate_df):,}")
print(f"Resolved binary candidates after tag/domain filtering: {len(candidate_df):,}")
display(candidate_df.head(15))


In [16]:
markets_df.columns.tolist()

## Step 4: Select One Demonstration Market

We scan the top resolved candidates by volume and keep the first one whose estimated trade history is large enough to be interesting but still practical for a notebook run.


In [8]:
selected_market = pick_demo_market(candidate_df, trades_collector)
selected_market


## Step 5: Download Full Historical Trades For The Selected Market

The trade history becomes the raw material for the `5m` probability panel.


In [9]:
trades_df = trades_collector.download_all_trades(
    str(selected_market["condition_id"]),
    frame_type="pandas",
    show_progress=True,
    estimate_total=True,
)

print(f"Downloaded trades: {len(trades_df):,}")
display(trades_df.head())


Normalizing trade size by /1000000.0 (size appears to be raw base units; applying 1e6 normalization).


Downloaded trades: 10,828


## Step 6: Convert Trade History Into A `5m` Yes-Probability Panel

For binary `Yes/No` markets we map every trade to a common `Yes` probability:

- `Yes` trade at price `p` -> `Yes probability = p`
- `No` trade at price `p` -> `Yes probability = 1 - p`

Then we resample to `5m` buckets and forward-fill the last observed probability so the result becomes a proper point-in-time panel.


In [10]:
probability_5m_df = build_yes_probability_series_5m(trades_df, selected_market)

print(f"5m rows: {len(probability_5m_df):,}")
display(probability_5m_df.head(12))
display(probability_5m_df.tail(12))


5m rows: 34,244


## Step 7: Save Metadata, Raw Trades, And The `5m` Panel Into SQLite

The notebook writes three tables:

- `selected_market_metadata`
- `selected_market_trades`
- `selected_market_probability_5m`


In [11]:
write_demo_sqlite(DB_PATH, selected_market, trades_df, probability_5m_df)

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        conn,
    )
    counts = {
        table: pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
        for table in tables["name"].tolist()
    }

print(f"Wrote SQLite database: {DB_PATH}")
display(tables)
counts


Wrote SQLite database: /Users/sneddy/research/polymarket_research/db/demo_resolved_probability_series.sqlite


## Step 8: Quick Sanity Check From SQLite

This is the final state we want for the main dataset direction: a database-backed, point-in-time `market x timestamp` panel that still retains raw trades and source metadata.


In [12]:
with sqlite3.connect(DB_PATH) as conn:
    sqlite_preview_df = pd.read_sql_query(
        """
        SELECT
            timestamp_utc,
            market_slug,
            yes_probability,
            trade_count,
            total_size,
            final_outcome
        FROM selected_market_probability_5m
        ORDER BY timestamp_utc
        LIMIT 20
        """,
        conn,
    )

display(sqlite_preview_df)
